# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² Mental Health Survey dataset from Kilifi County, Kenya, using the `mlcroissant` library. All data entities (record sets, fields, columns) are referenced by their `@id` as per Croissant best practices.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Initialize and load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a Metadata object
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their fields. All entities are referenced by their `@id`.

Let's print the available record sets, their IDs, and their fields for reference.

In [ ]:
# List all record sets, their fields, and associated field IDs
record_sets = dataset.metadata.record_sets
print(f"Found {len(record_sets)} record sets\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}\n  Name: {rs.name}\n  Description: {rs.description}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}) [dataType: {field.data_type}] | Description: {field.description}")
    print('---')

## 3. Data Extraction
Load data from all available record sets. Data is loaded into pandas DataFrames, with the key as the record set `@id`.

In [ ]:
dataframes = {}

# Iterate through all record sets by @id to extract data
for rs in dataset.metadata.record_sets:
    rs_id = rs.id
    # Extract all records from this record set
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded record set: {rs.name} (@id: {rs_id}) -- shape: {df.shape}")
    print(f"  Columns: {df.columns.tolist() if not df.empty else 'No records'}")
    print('---')

## 4. Exploratory Data Analysis (EDA)
Let's select one main record set (typically the survey or tabular data) and perform basic operations: filter by a numeric field, normalize, and group by a demographic attribute.

We'll use the following steps (customize field `@id`s as needed based on the overview above):
- Select a **numeric field** (e.g., PHQ-9, GAD-7 or PSQ score column)
- Select a **grouping field** (e.g., `@id` for gender, age_group, etc.)
- Filter, normalize, and group the data

In [ ]:
# Identify main record set for EDA (choose the most populated one)
main_rs = None
max_rows = 0
for rs_id, df in dataframes.items():
    if df.shape[0] > max_rows:
        main_rs = rs_id
        max_rows = df.shape[0]

print(f"Using record set '{main_rs}' for EDA.\n")

# Inspect columns and let the user pick a numeric field and group field
main_df = dataframes[main_rs]
print("Columns:", main_df.columns.tolist())

# Common survey fields: e.g. PHQ-9, GAD-7, PSQ Score, Age, etc. (use correct ID as per the schema)
# Replace the string below with the actual @id of the desired field if needed.
numeric_field = None
for col in main_df.columns:
    low_col = col.lower()
    if ('phq' in low_col or 'gad' in low_col or 'psq' in low_col or 'age' == low_col or 'score' in low_col) and pd.api.types.is_numeric_dtype(main_df[col]):
        numeric_field = col
        break
if numeric_field is None:
    # fallback: take the first numeric column
    for col in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field = col
            break

print(f"Selected numeric field for EDA: {numeric_field}\n")

# Grouping field: e.g., Gender, Age group, Education level
group_field = None
for col in main_df.columns:
    if col.lower() in ['gender', 'sex', 'age_group', 'level_of_education', 'marital_status']:
        group_field = col
        break
if group_field is None and main_df.columns.size:
    # fallback pick first non-numeric
    for col in main_df.columns:
        if not pd.api.types.is_numeric_dtype(main_df[col]):
            group_field = col
            break
print(f"Selected grouping field for EDA: {group_field}\n")

#--- EDA: Filter, Normalize, Group ---
if numeric_field and numeric_field in main_df.columns:
    # Example threshold: median
    threshold = main_df[numeric_field].median()
    filtered_df = main_df[main_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (median): {len(filtered_df)} rows\n")
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Group and aggregate
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean {numeric_field} by {group_field}:")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and illustrate grouping by the chosen demographic (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if numeric_field and numeric_field in main_df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field and group_field in main_df.columns:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=main_df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

- This notebook demonstrated loading, exploring, and processing the FAIR² mental health survey data via Croissant and `mlcroissant`.
- Entities were referenced consistently by their `@id` field, ensuring reproducibility.
- Data preview, normalization, filtering, grouping, and basic visualizations were shown.
- For further exploration, refer to the detailed [Croissant schema](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json) and modify field IDs as needed for domain-specific analysis.
